In [ ]:
from utils import *

import torch as th
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.nn.functional as F

from sklearn.model_selection import KFold

import os
import json
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms


import tqdm



In [ ]:
# Paths
img_dir = 'data/Databases/Quality/'
json_path = 'data/Databases/Quality/labels.json'

class ImageDataset(Dataset):
    def __init__(self, img_dir, json_path, transform=None, augment=True):
        self.img_dir = img_dir
        self.transform = transform
        self.augment = augment
        self.img_names = os.listdir(img_dir)
        self.img_names = [name for name in self.img_names if name != "labels.json"]
        self.num_images = len(self.img_names)

        with open(json_path, 'r') as f:
            self.labels_json = json.load(f)

        original_labels = list(self.labels_json["CleanCat"].values())
        unique_labels = sorted(set(original_labels))
        label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
        self.labels = [label_to_idx[label] for label in original_labels]

        self.num_classes = len(unique_labels)

        # Debug prints to verify label mapping
        print(f"Unique labels: {unique_labels}")
        print(f"Label to index mapping: {label_to_idx}")
        print(f"Mapped labels: {self.labels[:10]}")
        print(f"Number of classes: {self.num_classes}")

        self.labels = list(self.labels_json["CleanCat"].values())

    def __len__(self):
        return self.num_images * 8 if self.augment else self.num_images

    def __getitem__(self, idx):
        img_idx = idx // 8
        augment_idx = idx % 8

        img_name = os.path.join(self.img_dir, self.img_names[img_idx])
        image = Image.open(img_name)

        rotation_angle = augment_idx * 45
        if rotation_angle != 0:
            image = image.rotate(rotation_angle)  # removed expand=True

        if self.transform:
            image = self.transform(image)

        label = self.labels[img_idx]
        return image, label


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
])

dataset = ImageDataset(img_dir, json_path, transform, augment=True)



# (80% train/val, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
dataset_tr, dataset_te = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))
print(len(dataset), len(dataset_tr), len(dataset_te))

"""import matplotlib.pyplot as plt
for i in range(8):
    image, label = dataset[i]
    image_np = image.permute(1, 2, 0).numpy()
    plt.subplot(2, 4, i+1)
    plt.imshow(image_np)
    plt.title(f"Rotation: {i*45}°")
    plt.axis('off')
plt.tight_layout()
plt.show()"""

# DataLoader (regular)
dataloader_tr = DataLoader(dataset=dataset_tr, shuffle=True, batch_size=8)
dataloader_te = DataLoader(dataset=dataset_te, shuffle=True, batch_size=8)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
    

class RNet18(nn.Module):
    def __init__(self, num_classes=4):
        super(RNet18, self).__init__()
        
        self.resnet = models.resnet18(weights="DEFAULT")
        
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.resnet(x)
    
    
class RNet50(nn.Module):
    def __init__(self, num_classes=4):
        super(RNet50, self).__init__()
        
        self.resnet = models.resnet50(weights="DEFAULT")
        
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.resnet(x)

# Size differences

In [ ]:
R18 = RNet18(num_classes=4)
R50 = RNet50(num_classes=4)

# Calculate parameter sizes
param_size_18 = sum(param.nelement() * param.element_size() for param in R18.parameters())
param_size_50 = sum(param.nelement() * param.element_size() for param in R50.parameters())

# Calculate buffer sizes
buffer_size_18 = sum(buffer.nelement() * buffer.element_size() for buffer in R18.buffers())
buffer_size_50 = sum(buffer.nelement() * buffer.element_size() for buffer in R50.buffers())

# Total sizes in megabytes
size_all_mb_18 = (param_size_18 + buffer_size_18) / (1024 ** 2)
size_all_mb_50 = (param_size_50 + buffer_size_50) / (1024 ** 2)

print('RNet18 size: {:.3f} MB'.format(size_all_mb_18))
print('RNet50 size: {:.3f} MB'.format(size_all_mb_50))


In [ ]:
# Pruning function
def apply_pruning(model, sparsity=0.5):
    """
    Applies unstructured pruning to each layer in the model.
    """
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            prune.l1_unstructured(module, name='weight', amount=sparsity)


In [ ]:
def train_and_prune_model(model, train_loader, criterion, optimizer, device, epochs, initial_sparsity=0.0, final_sparsity=0.9):
    """
    Gradually prunes the model over several epochs, applying pruning before forward and backward passes.
    """
    sparsity_step = (final_sparsity - initial_sparsity) / epochs

    for epoch in range(epochs):
        current_sparsity = initial_sparsity + epoch * sparsity_step

        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()

            # Apply pruning before forward pass
            apply_pruning(model, sparsity=current_sparsity)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Apply pruning before backward pass
            apply_pruning(model, sparsity=current_sparsity)

            # Backward pass
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Sparsity: {current_sparsity:.2f}, Train Loss: {avg_train_loss:.4f}")

    # Remove pruning re-parametrization to finalize the model's sparsity
    for module in model.modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            prune.remove(module, 'weight')

    # Fine-tune the pruned model
    fine_tune_epochs = 5  
    for epoch in range(fine_tune_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        avg_fine_tune_loss = running_loss / len(train_loader)
        print(f"Fine-tuning Epoch {epoch+1}/{fine_tune_epochs}, Loss: {avg_fine_tune_loss:.4f}")

    return model

In [ ]:
def validate_model(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    return running_loss / len(val_loader), accuracy

In [ ]:
size = "18"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# KFold split 
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True)

# Parameters
epochs = 5
lr = 0.0001
initial_sparsity = 0.0
final_sparsity = 0.30


pruned=True

# Cross-validation loop
for fold, (train_idx, val_idx) in enumerate(kf.split(dataset_tr)):
    print(f'Fold {fold+1}/{k_folds}')
    
    # Create subsets for this fold
    train_subset = Subset(dataset_tr, train_idx)
    val_subset = Subset(dataset_tr, val_idx)
    
    # Create DataLoader for train and validation
    train_loader = DataLoader(train_subset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=4, shuffle=True)
    
    # Initialize the model, loss function, and optimizer for each fold
    if size == "18":
        model = RNet18(num_classes=4).to(device)
    else:
        model = RNet50(num_classes=4).to(device)


    loss_func = nn.CrossEntropyLoss()
    optimizer = th.optim.Adam(model.parameters(), lr=lr)
    best_val_accuracy = 0.0
    model = train_and_prune_model(model, train_loader, loss_func, optimizer, device, epochs, initial_sparsity, final_sparsity)

    # Validation phase after pruning
    val_loss, val_accuracy = validate_model(model, val_loader, loss_func, device)
    print(f'Validation after pruning - Fold {fold+1}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')
    
    # Save the model if validation accuracy improves
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        model_file = f'pruned_RNet{size}_fold_{fold+1}_{int(initial_sparsity*100)}_{int(final_sparsity*100)}.pth'
        if pruned:
            torch.save(model.state_dict(), os.path.join("trained_models/quality/pruned/", model_file))
        else:
            torch.save(model.state_dict(), os.path.join("trained_models/quality/", model_file))
        print(f'Model saved for fold {fold+1} with val accuracy {val_accuracy:.4f}')

    print(f'Fold {fold+1} finished.\n')

print('Cross-validation complete.')

In [ ]:
size = size

# Load the model
if size=="18":
    model = RNet18(num_classes=4).to(device)
else:
    model = RNet50(num_classes=4).to(device)
    
model.load_state_dict(th.load(f"trained_models/quality/pruned/pruned_RNet{size}_fold_5_0_30.pth"))
#model.load_state_dict(th.load(f"trained_models/quality/RNet18_5fold_2_np.pth"))

# Set the model to evaluation mode
model.eval()

# Test loop
total_images = 0
nr_acc = 0

with th.no_grad():
    for images, labels in dataloader_te:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        
        # Predicted classes
        predicted = th.argmax(outputs, dim=1)
        
        # Accuracy calculation
        nr_acc += (predicted == labels).sum().item()
        total_images += labels.size(0)
        
        """if total_images <= 10:
            plt.imshow(images[0].cpu().permute(1, 2, 0).numpy())
            plt.title(f"Pred: {predicted[0].item()} | Label: {labels[0].item()}")
            plt.axis('off')
            plt.show()"""
    
print(f"Test Accuracy: {nr_acc / total_images:.4f}")
